In [8]:
from pathlib import Path
import json

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = Path.home() / "projects/bybit-history-downloader"

MERGED = {
    mode: ROOT / "data/merged" / f"final-{mode}"
    for mode in ("trades", "depth", "combined")
}

manifests = {}

for mode, path in MERGED.items():
    manifest_path = path / "manifest.json"

    assert manifest_path.is_file(), f"Missing manifest: {manifest_path}"

    manifests[mode] = json.loads(manifest_path.read_text())

    print(
        f"{mode:10} | "
        f"events={manifests[mode]['events_written']:,} | "
        f"files={manifests[mode]['files_written']} | "
        f"job={manifests[mode]['job_id']}"
    )

trades     | events=50,106,717 | files=48 | job=930001
depth      | events=405,581,426 | files=48 | job=930002
combined   | events=455,688,143 | files=55 | job=930003


In [9]:
summary = []

for mode, manifest in manifests.items():
    summary.append({
        "mode": mode,
        "job_id": manifest.get("job_id"),
        "protocol_version": manifest.get("protocol_version"),
        "manifest_mode": manifest.get("mode"),
        "input_datasets": len(manifest.get("input_datasets", [])),
        "events": manifest.get("events_written"),
        "trade_events": manifest.get("trade_events"),
        "depth_events": manifest.get("depth_events"),
        "files": manifest.get("files_written"),
        "skipped_initializations": manifest.get("skipped_initializations"),
    })

display(pl.DataFrame(summary))

mode,job_id,protocol_version,manifest_mode,input_datasets,events,trade_events,depth_events,files,skipped_initializations
str,i64,i64,str,i64,i64,i64,i64,i64,list[struct[6]]
"""trades""",930001,1,"""trades""",21,50106717,50106717,0,48,[]
"""depth""",930002,1,null,15,405581426,null,null,48,"[{96,1,44067,0,44067,1788224400000000000}, {103,1,1726,0,1726,1788224400000000000}, … {112,71,53531,0,53531,1788476400000000000}]"
"""combined""",930003,1,"""combined""",36,455688143,50106717,405581426,55,"[{96,1,44067,0,44067,1788224400000000000}, {103,1,1726,0,1726,1788224400000000000}, … {112,71,53531,0,53531,1788476400000000000}]"


In [13]:
schemas = {}

for mode, path in MERGED.items():
    files = sorted(path.rglob("*.parquet"))

    assert files, f"No Parquet files found: {path}"

    parquet = pq.ParquetFile(files[0])
    schemas[mode] = parquet.schema_arrow

    print(f"\n{'=' * 80}")
    print(f"{mode.upper()} PHYSICAL SCHEMA")
    print(f"{'=' * 80}")
    print(schemas[mode])


TRADES PHYSICAL SCHEMA
event_timestamp_ns: int64 not null
system_timestamp_ns: int64
exchange: string not null
instrument_id: int64 not null
symbol: string not null
stream_id: string not null
trade_id: string
sequence: uint64
side: string not null
price: decimal256(76, 28) not null
quantity_base: decimal256(76, 28)
quantity_quote: decimal256(76, 28)
quantity_contracts: decimal256(76, 28)
is_rpi: bool
trade_iv: decimal256(76, 28)
mark_iv: decimal256(76, 28)
index_price: decimal256(76, 28)
mark_price: decimal256(76, 28)
stream_rank: uint32 not null
dataset_id: uint64 not null
task_id: uint64 not null
event_ordinal: uint64 not null
canonical_row_offset: uint64 not null
canonical_row_count: uint64 not null

DEPTH PHYSICAL SCHEMA
event_timestamp_ns: int64 not null
stream_rank: uint32 not null
dataset_id: uint64 not null
task_id: uint64 not null
event_ordinal: uint64 not null
canonical_row_offset: uint64 not null
canonical_row_count: uint64 not null
event_boundary: uint8 not null
source_ope

In [14]:
samples = {}

for mode, path in MERGED.items():
    files = sorted(path.rglob("*.parquet"))

    parquet = pq.ParquetFile(files[0])
    batch = next(parquet.iter_batches(batch_size=10), None)

    assert batch is not None, f"Empty Parquet file: {files[0]}"

    samples[mode] = pa.Table.from_batches([batch])

    print(f"\n{'=' * 80}")
    print(f"{mode.upper()} SAMPLE")
    print(f"{'=' * 80}")

    display(samples[mode].to_pandas())


TRADES SAMPLE


,event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,trade_id,sequence,side,price,...,trade_iv,mark_iv,index_price,mark_price,stream_rank,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count
0,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470684897984512,NaN,sell,78552.8000000000000000000000000000,...,None,None,None,None,3,93,2,62633,62633,1
1,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470684897984514,NaN,sell,78552.8000000000000000000000000000,...,None,None,None,None,3,93,2,62634,62634,1
2,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470685208363008,NaN,sell,78552.8000000000000000000000000000,...,None,None,None,None,3,93,2,62635,62635,1
3,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470685220945920,NaN,sell,78552.8000000000000000000000000000,...,None,None,None,None,3,93,2,62636,62636,1
4,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470685220945922,NaN,sell,78552.8000000000000000000000000000,...,None,None,None,None,3,93,2,62637,62637,1
5,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470685220945924,NaN,sell,78551.2000000000000000000000000000,...,None,None,None,None,3,93,2,62638,62638,1
6,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470686437294080,NaN,sell,78550.4000000000000000000000000000,...,None,None,None,None,3,93,2,62639,62639,1
7,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470686437294082,NaN,sell,78550.4000000000000000000000000000,...,None,None,None,None,3,93,2,62640,62640,1
8,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470686437294084,NaN,sell,78550.4000000000000000000000000000,...,None,None,None,None,3,93,2,62641,62641,1
9,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,1478470686437294086,NaN,sell,78550.4000000000000000000000000000,...,None,None,None,None,3,93,2,62642,62642,1



DEPTH SAMPLE


,event_timestamp_ns,stream_rank,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation
0,1788220800000000000,3,96,0,44061,0,44061,1,0
1,1788220800000000000,3,96,0,44062,44061,1,0,2
2,1788220800000000000,3,96,0,44063,44062,1,0,2
3,1788220800000000000,3,96,0,44064,44063,1,0,2
4,1788220800000000000,3,96,0,44065,44064,1,0,2
5,1788220800000000000,3,96,0,44066,44065,1,0,2
6,1788220800000000000,3,96,0,44067,44066,1,0,2
7,1788220800000000000,3,96,0,44068,44067,1,0,2
8,1788220800000000000,3,96,0,44069,44068,1,0,2
9,1788220800000000000,3,96,0,44070,44069,1,0,2



COMBINED SAMPLE


,event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,event_kind,stream_rank,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,trade,depth
0,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62633,62633,1,"{'trade_id': '1478470684897984512', 'sequence'...",None
1,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62634,62634,1,"{'trade_id': '1478470684897984514', 'sequence'...",None
2,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62635,62635,1,"{'trade_id': '1478470685208363008', 'sequence'...",None
3,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62636,62636,1,"{'trade_id': '1478470685220945920', 'sequence'...",None
4,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62637,62637,1,"{'trade_id': '1478470685220945922', 'sequence'...",None
5,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62638,62638,1,"{'trade_id': '1478470685220945924', 'sequence'...",None
6,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62639,62639,1,"{'trade_id': '1478470686437294080', 'sequence'...",None
7,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62640,62640,1,"{'trade_id': '1478470686437294082', 'sequence'...",None
8,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62641,62641,1,"{'trade_id': '1478470686437294084', 'sequence'...",None
9,1788220800000000000,NaN,bitget,14215,BTCUSDT,bitget:perpetual:linear:BTCUSDT:trade,0,3,93,2,62642,62642,1,"{'trade_id': '1478470686437294086', 'sequence'...",None


In [15]:
from pathlib import Path
import pyarrow.parquet as pq

for mode, root in MERGED.items():
    print(f"\n{'=' * 80}")
    print(f"{mode.upper()} DATASET STRUCTURE")
    print("=" * 80)

    parquet_files = sorted(root.rglob("*.parquet"))

    groups = {}

    for path in parquet_files:
        relative = path.relative_to(root)
        category = relative.parts[0]

        groups.setdefault(category, []).append(path)

    for category, files in groups.items():
        print(f"\n{category}/ — {len(files)} files")

        example = files[0]
        print(f"Example: {example.relative_to(root)}")
        print(f"Schema:\n{pq.read_schema(example)}")


TRADES DATASET STRUCTURE

timeline/ — 48 files
Example: timeline/partition-000000/part-000000.parquet
Schema:
event_timestamp_ns: int64 not null
system_timestamp_ns: int64
exchange: string not null
instrument_id: int64 not null
symbol: string not null
stream_id: string not null
trade_id: string
sequence: uint64
side: string not null
price: decimal256(76, 28) not null
quantity_base: decimal256(76, 28)
quantity_quote: decimal256(76, 28)
quantity_contracts: decimal256(76, 28)
is_rpi: bool
trade_iv: decimal256(76, 28)
mark_iv: decimal256(76, 28)
index_price: decimal256(76, 28)
mark_price: decimal256(76, 28)
stream_rank: uint32 not null
dataset_id: uint64 not null
task_id: uint64 not null
event_ordinal: uint64 not null
canonical_row_offset: uint64 not null
canonical_row_count: uint64 not null

DEPTH DATASET STRUCTURE

timeline/ — 48 files
Example: timeline/partition-000000/part-000000.parquet
Schema:
event_timestamp_ns: int64 not null
stream_rank: uint32 not null
dataset_id: uint64 not nul

In [16]:
for mode, manifest in manifests.items():
    print(f"\n{mode.upper()}")

    for entry in manifest["files"][:5]:
        print(entry["path"])


TRADES
timeline/partition-000000/part-000000.parquet
timeline/partition-000001/part-000000.parquet
timeline/partition-000002/part-000000.parquet
timeline/partition-000003/part-000000.parquet
timeline/partition-000004/part-000000.parquet

DEPTH
timeline/partition-000000/part-000000.parquet
timeline/partition-000001/part-000000.parquet
timeline/partition-000002/part-000000.parquet
timeline/partition-000003/part-000000.parquet
timeline/partition-000004/part-000000.parquet

COMBINED
timeline/partition-000000/part-000000.parquet
timeline/partition-000001/part-000000.parquet
timeline/partition-000002/part-000000.parquet
timeline/partition-000003/part-000000.parquet
timeline/partition-000004/part-000000.parquet


In [17]:
from pathlib import Path
import pyarrow.parquet as pq

depth_path = ROOT / "data/merged/final-depth-v2"

files = sorted(depth_path.rglob("*.parquet"))

print("Files:", len(files))
print("Schema:")
print(pq.read_schema(files[0]))

parquet = pq.ParquetFile(files[0])
batch = next(parquet.iter_batches(batch_size=5))

display(batch.to_pandas())

Files: 53
Schema:
event_timestamp_ns: int64 not null
system_timestamp_ns: int64
exchange: string not null
instrument_id: int64 not null
symbol: string not null
stream_id: string not null
stream_rank: uint32 not null
dataset_id: uint64 not null
task_id: uint64 not null
event_ordinal: uint64 not null
canonical_row_offset: uint64 not null
canonical_row_count: uint64 not null
event_boundary: uint8 not null
source_operation: uint8 not null
sequence_start: uint64
sequence_end: uint64
changes: list<item: struct<side: string not null, price: decimal256(76, 28) not null, quantity_base: decimal2 (... 118 chars omitted) not null
  child 0, item: struct<side: string not null, price: decimal256(76, 28) not null, quantity_base: decimal256(76, 28), (... 97 chars omitted) not null
      child 0, side: string not null
      child 1, price: decimal256(76, 28) not null
      child 2, quantity_base: decimal256(76, 28)
      child 3, quantity_quote: decimal256(76, 28)
      child 4, quantity_contracts: dec

,event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,stream_rank,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation,sequence_start,sequence_end,changes
0,1788220800000000000,NaN,gateio,17397,BTC_USDT,gateio:perpetual:linear:BTC_USDT:l2,3,96,0,44061,0,44061,1,0,123743115561,123743115561,"[{'side': 'bid', 'price': 0.100000000000000000..."
1,1788220800000000000,NaN,gateio,17397,BTC_USDT,gateio:perpetual:linear:BTC_USDT:l2,3,96,0,44062,44061,1,0,2,123743115562,123743115562,"[{'side': 'bid', 'price': 60863.50000000000000..."
2,1788220800000000000,NaN,gateio,17397,BTC_USDT,gateio:perpetual:linear:BTC_USDT:l2,3,96,0,44063,44062,1,0,2,123743115563,123743115563,"[{'side': 'bid', 'price': 78542.10000000000000..."
3,1788220800000000000,NaN,gateio,17397,BTC_USDT,gateio:perpetual:linear:BTC_USDT:l2,3,96,0,44064,44063,1,0,2,123743115564,123743115564,"[{'side': 'ask', 'price': 79283.00000000000000..."
4,1788220800000000000,NaN,gateio,17397,BTC_USDT,gateio:perpetual:linear:BTC_USDT:l2,3,96,0,44065,44064,1,0,2,123743115565,123743115565,"[{'side': 'ask', 'price': 79314.80000000000000..."
